**Import Required Libraries**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

**Load Project Utilities & Initialize Notebook Widgets**

In [0]:
%run /Workspace/Users/mohittaneja222@gmail.com/1_dev_food_chain_project/1_Setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "goods_ct", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f'/Volumes/goods_ct/bronze/vol_goods/child_company/full_load/{data_source}/{data_source}.csv'
print(base_path)


## Bronze

In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
            .mode("overwrite")\
                .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver


In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

**Transformations**

- 1: Drop Duplicates

In [0]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['product_id'])
print('Rows after duplicates dropped: ', df_silver.count())

- 2: Title case fix

(energy bars ---> Energy Bars, protien bars ---> Protien Bars etc)

In [0]:
df_silver.select('category').distinct().show()

In [0]:
#Title Case Fix
df_silver = df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(), None)
    .otherwise(F.initcap("category"))
)

In [0]:
df_silver.select('category').distinct().show()

- 3: Fix Spelling Mistake for `Protien`

In [0]:
df_silver = (
    df_silver.withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Protein", "Protein"))
    .withColumn("category", F.regexp_replace(F.col("category"), "(?i)Protien", "Protein"))
   )

In [0]:
display(df_silver.limit(5))

### Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
##1. Add division column
df_silver = (
    df_silver.withColumn("division",
            F.when(F.col("category") == "Energy Bars", "Nutrition Bars")
             .when(F.col("category") == "Protein bars", "Nutrition bars")
             .when(F.col("category") == "Granola & cereals", "Breakfast Foods")
             .when(F.col("category") == "Recovery Dairy", "Dairy & Recovery")
             .when(F.col("category") == "Healthy Snacks", "Healthy Snacks")
             .when(F.col("category") == "Electrolyte Mix", "Hydration & Electrolytes")
             .otherwise("Other")
    )
)

##2: Variant Column
df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)

##3. Adding new Columns : product code
df_silver = (
    df_silver
        .withColumn("product_code", F.sha2(F.col("product_name"), 256))
        .withColumn("product_id", 
                    F.when(F.col("product_id").cast("string").rlike("^[0-9]+$"),
                           F.col("product_id").cast("string")
                    ).otherwise(F.lit(999999).cast("string"))
        )
        .withColumnRenamed("product_name", "product")
)

In [0]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size")

In [0]:
display(df_silver)

In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")
df_gold.show(10)

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sh_dim_{data_source}")

## Merging Data source with parent

In [0]:
delta_table = DeltaTable.forName(spark,"goods_ct.gold.dim_products")
df_child_products = spark.sql(f"select product_code, division, category, product, variant FROM goods_ct.gold.sh_dim_products;")
df_child_products.show(5)


In [0]:
delta_table.alias("target").merge(
    source= df_child_products.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()